In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 80)

## 1) Carregamento do Corpus

In [2]:
with open("manifestacoes.json", encoding="utf-8") as f:
    manifestacoes = json.load(f)

df = pd.DataFrame(manifestacoes)
id_para_idx = {row.id: i for i, row in df.iterrows()}
corpus = df["texto"].tolist()

print(f"Total de manifestações: {len(df)}")
df.head()

Total de manifestações: 40


,id,data,categoria_oficial,texto
0,M001,2026-02-03,infraestrutura,"Falta iluminação pública na Rua das Acácias há mais de um mês, moradores rel..."
1,M002,2026-02-05,infraestrutura,Bueiro aberto na esquina da Rua XV de Novembro representa risco grave para p...
2,M003,2026-02-07,infraestrutura,"Existe um buraco enorme na Av. Brasil, próximo ao número 450, que já danific..."
3,M017,2026-03-02,infraestrutura,O asfalto todo esburacado da avenida principal está causando acidentes const...
4,M004,2026-02-10,infraestrutura,Calçada quebrada em frente à escola municipal dificulta a passagem de cadeir...


## 2) Representação Esparsa: Bag-of-Words
Conta a frequência bruta de cada termo, sem considerar ordem ou peso relativo dos termos.

In [3]:
bow = CountVectorizer()
X_bow = bow.fit_transform(corpus)
print(f"Vocabulário BoW: {len(bow.vocabulary_)} termos")
print(f"Matriz BoW: {X_bow.shape} (manifestações × termos)")

Vocabulário BoW: 552 termos
Matriz BoW: (40, 552) (manifestações × termos)


## 3) Representação Esparsa: TF-IDF
Pondera os termos pela raridade no corpus (IDF), reduzindo a importância de palavras muito comuns.

In [4]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)
print(f"Matriz TF-IDF: {X_tfidf.shape}")

Matriz TF-IDF: (40, 552)


## 4) Representação Densa: Embeddings
Usamos um modelo multilíngue (`paraphrase-multilingual-MiniLM-L12-v2`), recomendado no enunciado por lidar melhor com o português brasileiro do que modelos treinados só em inglês.

In [5]:
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
X_emb = modelo.encode(corpus, show_progress_bar=True)
print(f"Matriz de Embeddings: {X_emb.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Matriz de Embeddings: (40, 384)


## 5) Comparação dos Pares Solicitados

In [6]:
def cosseno(matriz, id_a, id_b):
    i, j = id_para_idx[id_a], id_para_idx[id_b]
    a, b = matriz[i], matriz[j]
    # normaliza para lidar tanto com matriz esparsa (scipy) quanto densa (numpy)
    a = a.toarray() if hasattr(a, "toarray") else a.reshape(1, -1)
    b = b.toarray() if hasattr(b, "toarray") else b.reshape(1, -1)
    return cosine_similarity(a, b)[0][0]

pares = [("M003", "M017", "Buraco/asfalto (duplicata esperada)"),
         ("M008", "M022", "Posto de saúde sem médico (duplicata esperada)"),
         ("M008", "M031", "Saúde × Iluminação (controle, temas diferentes)")]

linhas = []
for id_a, id_b, descricao in pares:
    linhas.append({
        "Par": f"{id_a} × {id_b}",
        "Descrição": descricao,
        "BoW": round(cosseno(X_bow, id_a, id_b), 4),
        "TF-IDF": round(cosseno(X_tfidf, id_a, id_b), 4),
        "Embeddings": round(cosseno(X_emb, id_a, id_b), 4),
    })

df_comparacao = pd.DataFrame(linhas)
df_comparacao

,Par,Descrição,BoW,TF-IDF,Embeddings
0,M003 × M017,Buraco/asfalto (duplicata esperada),0.0000,0.0000,0.4987
1,M008 × M022,Posto de saúde sem médico (duplicata esperada),0.1777,0.1694,0.6395
2,M008 × M031,"Saúde × Iluminação (controle, temas diferentes)",0.1185,0.0547,0.1370


In [7]:
for id_a, id_b, _ in pares:
    print(f"{id_a}: {df.loc[id_para_idx[id_a], 'texto']}")
    print(f"{id_b}: {df.loc[id_para_idx[id_b], 'texto']}")
    print("-" * 100)

M003: Existe um buraco enorme na Av. Brasil, próximo ao número 450, que já danificou pneus de vários carros.
M017: O asfalto todo esburacado da avenida principal está causando acidentes constantes com motociclistas.
----------------------------------------------------------------------------------------------------
M008: O posto de saúde do bairro Esperança está sem médico há duas semanas, pacientes crônicos ficaram sem atendimento.
M022: Está faltando atendimento médico no PSF da nossa região, os moradores não conseguem marcar consulta.
----------------------------------------------------------------------------------------------------
M008: O posto de saúde do bairro Esperança está sem médico há duas semanas, pacientes crônicos ficaram sem atendimento.
M031: A lâmpada queimada na praça do bairro deixa o local totalmente escuro à noite, o que preocupa os moradores.
----------------------------------------------------------------------------------------------------


## 6) Discussão dos Resultados

Preencha esta seção depois de rodar o notebook, usando os números da tabela `df_comparacao`
acima. Um roteiro para guiar a análise:

- **M003 × M017:** os dois textos falam do mesmo problema (buraco/asfalto danificado),
  mas com vocabulário quase totalmente diferente ("buraco" vs. "esburacado", "Av. Brasil" vs.
  "avenida principal"). Espera-se que **BoW e TF-IDF fiquem baixos** (pouca sobreposição
  lexical) e que os **Embeddings fiquem visivelmente mais altos**, por capturarem que as duas
  frases descrevem a mesma situação semântica.
- **M008 × M022:** mesmo padrão do par anterior — "posto de saúde sem médico" e "falta
  atendimento no PSF" compartilham pouco vocabulário exato, mas o significado é equivalente.
  Novamente, espera-se um salto grande de BoW/TF-IDF para Embeddings.
- **M008 × M031:** as duas manifestações tratam de assuntos diferentes (saúde vs. iluminação
  pública). Aqui, mesmo os Embeddings devem apresentar similaridade **baixa**, o que serve como
  "prova de sanidade" de que o modelo não está apenas atribuindo scores altos para qualquer par.

Compare os valores obtidos com esse roteiro: se M003×M017 e M008×M022 vierem com Embeddings
sensivelmente maiores que BoW/TF-IDF, e M008×M031 vier baixo nas três representações, os
resultados confirmam a hipótese central do desafio — representações densas capturam
sinônimos e paráfrases que representações lexicais não conseguem enxergar.

## 7) Conclusão — Limitações de Cada Abordagem

**Bag-of-Words** é a representação mais simples e rápida de calcular, mas ignora completamente
a ordem das palavras e o significado: duas frases sinônimas com vocabulário diferente (como
"buraco" e "esburacado") são tratadas como não relacionadas, e frases com as mesmas palavras em
ordens muito diferentes são tratadas como idênticas. Além disso, gera vetores extremamente
esparsos e de alta dimensão, crescendo com o vocabulário do corpus.

**TF-IDF** melhora o BoW ao penalizar termos muito frequentes (como "posto", "saúde", "rua",
que aparecem em várias manifestações) e valorizar termos mais discriminativos, mas continua
sendo uma representação puramente lexical: não resolve o problema de sinônimos, não entende
negação, e ainda sofre com o problema de dimensionalidade e esparsidade do BoW.

**Embeddings densos** capturam relações semânticas e sinônimos, permitindo detectar que duas
manifestações com vocabulário totalmente diferente descrevem o mesmo problema — exatamente o
caso de uso da Ouvidoria. As limitações aqui são outras: exigem mais poder computacional para
gerar e comparar (principalmente em corpus grandes), dependem fortemente da qualidade e do
idioma do modelo pré-treinado escolhido (por isso a preferência por modelos multilíngues para
português), e são menos interpretáveis — não é possível apontar "qual palavra" gerou a
similaridade, como é possível no TF-IDF.